# 03 – XGBoost Training, Evaluation & Deployment Preparation
End-to-end training notebook for the final production model.

## Objectives
- Train the final XGBoost model
- Tune hyperparameters
- Evaluate performance
- Analyze errors
- Export a deployment-ready pipeline

In [2]:
import pandas as pd
import numpy as np
import joblib
import json 

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

import matplotlib.pyplot as plt


In [3]:
df_model = pd.read_csv("../data/processed.csv", dtype={"fuel_type": str})

df_model.to_parquet("data.parquet", engine = "pyarrow")

df_model = pd.read_parquet("data.parquet", engine = "pyarrow", )

df_model.dtypes

price             float64
miles             float64
year              float64
make                  str
body_type             str
vehicle_type          str
drivetrain            str
transmission          str
fuel_type             str
engine_size       float64
engine_block          str
city                  str
province              str
miles_was_zero      int64
car_age           float64
miles_per_year    float64
model_trim            str
region                str
dtype: object

Pasting code from N02 -> Preprocessing Pipelines, defining X and y, target and predictor variables as well Column transformer

In [4]:
TARGET = "price"

X = df_model.drop(columns=[TARGET]).copy()
y = df_model[TARGET].copy()

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

In [6]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()


In [7]:
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

X[categorical_features].nunique().sort_values(ascending=False)

model_trim      5998
region           963
city             758
make              57
fuel_type         24
body_type         21
province          15
engine_block       3
drivetrain         3
vehicle_type       2
transmission       2
dtype: int64

In [8]:
high_cardinality_features = [
    "model_trim",
    "region",
    "city"
]

In [9]:
cat_columns_to_drop = [
    'model_trim' , 'region' , 'city' ,
    'model', 'trim'
]

low_cardinality_features = [
    col for col in categorical_cols
    if col not in cat_columns_to_drop
]

# High-cardinality categorical columns with separate transformers
model_trim_features = ["model_trim"]
region_features = ["region"]
city_features = ["city"]

# Final selected features
selected_features = (
    numeric_cols
    + low_cardinality_features
    + model_trim_features
    + region_features
    + city_features
)

selected_features = list(dict.fromkeys(selected_features))

# Remove accidental duplicates
selected_features = list(dict.fromkeys(selected_features))

In [10]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]
)

low_cardinality_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#
model_trim_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])

region_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

city_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numeric_transformer, numeric_cols),

    ("low_cardinality_categorical",
     low_cardinality_categorical_transformer,
     low_cardinality_features),

    ("model_trim",
     model_trim_transformer,
     model_trim_features),

    ("region",
     region_transformer,
     region_features),

    ("city",
     city_transformer,
     city_features)
])

preprocessor 

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('low_cardinality_categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_

After copy-pasting the code it is now time to define the model

Simple baseline model before performing serious hyperparameter tuning

In [11]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return {

        "MAE" : mean_absolute_error(y, pred),
        "R2" : r2_score(y, pred),
        "RMSE" : np.sqrt(mean_squared_error(y, pred)), 

    }

baseline_xgb = XGBRegressor(
        n_estimators=1500,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=1,
    gamma=0,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1,
    reg_alpha=0,
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
)

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_xgb)
])

baseline_pipeline.fit(X_train, y_train)
baseline_metrics = evaluate(baseline_pipeline, X_test, y_test)
print("BASELINE (beat this):", baseline_metrics)
BASELINE_MAE = baseline_metrics["MAE"]

c:\Users\alecf\Code\2025-2026\used_car_app\.venv\Lib\site-packages\xgboost\core.py:553: UserWarning: [23:20:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


BASELINE (beat this): {'MAE': 2738.722653593595, 'R2': 0.9356699216410026, 'RMSE': np.float64(5140.382981240645)}


## Hyperparameter Tuning 

Goal is to initialize a set of weights, and parameters in order to minimize total size but also maximize model performance. I will then perform error analysis to determine whether the model overfit or underfit.

In [ ]:
from sklearn.model_selection import GridSearchCV


CV_FOLDS = 3
SCORING = "neg_mean_absolute_error"

def make_pipeline(**xgb_overrides):

    params = dict(
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1,
    )
    params.update(xgb_overrides)
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**params)),
    ])

def run_grid(param_grid, fixed_params, stage_name):
   
    pipe = make_pipeline(**fixed_params)
    grid = {f"model__{k}": v for k, v in param_grid.items()}
    search = GridSearchCV(
        pipe,
        param_grid=grid,
        scoring=SCORING,
        cv=CV_FOLDS,
        n_jobs=1,        
        verbose=2,
        refit=True,
    )
    print(f"=== {stage_name} ===")
    search.fit(X_train, y_train)
    print("Best CV MAE:", -search.best_score_)
    print("Best params:", search.best_params_)
    return search
